In [4]:
import polars as pl

trades = pl.read_parquet("../parquets/H_gateio_btcusdt_perp_trades.parquet").with_columns(
    pl.from_epoch("ingress_ts", time_unit="us"),
    pl.from_epoch("transaction_ts", time_unit="us"),
    pl.from_epoch("publish_ts", time_unit="us"),
)

static = pl.read_parquet("../parquets/H_gateio_btcusdt_perp_instrument_static.parquet").with_columns(
    pl.from_epoch("ingress_ts", time_unit="us"),
    pl.from_epoch("publish_ts", time_unit="us"),
)

In [5]:
trades.select(pl.col("qty").sum().alias("native_contracts"))

native_contracts
f64
8.959318e6


In [6]:
quantity_multiplier = static.select(pl.col("quantity_multiplier").fill_null(1)).item()

trades.select(
    (pl.col("qty") * quantity_multiplier).sum().alias("base_asset_btc")
)

base_asset_btc
f64
895.9318


In [7]:
trades.select(
    (pl.col("qty") * quantity_multiplier * pl.col("price")).sum().alias("quote_usdt")
)

quote_usdt
f64
6.6923e7


In [9]:
import requests

public = requests.get(
    "https://api.gateio.ws/api/v4/futures/usdt/candlesticks",
    params={
        "contract": "BTC_USDT",
        "from": 1779537600,
        "to": 1779541200,
        "interval": "1h",
    },
).json()

public

[{'o': '74750.4',
  'v': 8959318,
  't': 1779537600,
  'c': '74722.3',
  'l': '74620',
  'h': '74762.2',
  'sum': '66923451.09224'},
 {'o': '74722.2',
  'v': 48302541,
  't': 1779541200,
  'c': '75172',
  'l': '74718.6',
  'h': '75210',
  'sum': '362289300.41438'}]